In [ ]:
from scipy.spatial import KDTree
import numpy as np

# Build KD-Tree for fast lookup of shape points
def build_kd_tree(route_df):
    points = route_df[['shape_pt_lat', 'shape_pt_lon']].values
    tree = KDTree(points)
    return tree, points

# Find the closest segment by searching for nearest point in the KD-Tree and its neighbors
def match_vehicle_to_route_kdtree(vehicle_df, route_df, kd_tree, points):
    matches = []

    for index, vehicle in vehicle_df.iterrows():
        vehicle_lat = vehicle['latitude']
        vehicle_lon = vehicle['longitude']

        # Query KD-Tree for nearest shape point
        dist, idx = kd_tree.query([vehicle_lat, vehicle_lon])
        closest_shape_point = points[idx]

        # Handle the line segment: Find neighbors of the closest point
        if idx > 0:
            prev_point = points[idx - 1]
        else:
            prev_point = None

        if idx < len(points) - 1:
            next_point = points[idx + 1]
        else:
            next_point = None

        # Check the closest point on the line segments
        closest_lat, closest_lon = closest_shape_point
        closest_dist = haversine(vehicle_lat, vehicle_lon, closest_lat, closest_lon)

        if prev_point is not None:
            prev_lat, prev_lon = prev_point
            proj_lat, proj_lon = closest_point_on_segment(vehicle_lat, vehicle_lon, prev_lat, prev_lon, closest_lat, closest_lon)
            dist_to_segment = haversine(vehicle_lat, vehicle_lon, proj_lat, proj_lon)
            if dist_to_segment < closest_dist:
                closest_lat, closest_lon = proj_lat, proj_lon
                closest_dist = dist_to_segment

        if next_point is not None:
            next_lat, next_lon = next_point
            proj_lat, proj_lon = closest_point_on_segment(vehicle_lat, vehicle_lon, closest_lat, closest_lon, next_lat, next_lon)
            dist_to_segment = haversine(vehicle_lat, vehicle_lon, proj_lat, proj_lon)
            if dist_to_segment < closest_dist:
                closest_lat, closest_lon = proj_lat, proj_lon
                closest_dist = dist_to_segment

        # Store the closest point and distance
        matches.append({
            'vehicle_id': vehicle['vehicle_id'],
            'vehicle_lat': vehicle_lat,
            'vehicle_lon': vehicle_lon,
            'closest_lat': closest_lat,
            'closest_lon': closest_lon,
            'distance_to_route': closest_dist
        })

    return pd.DataFrame(matches)

# Assuming you have a route DataFrame with shape points and a vehicle DataFrame
kd_tree, points = build_kd_tree(route_df)
matched_df = match_vehicle_to_route_kdtree(vehicle_df, route_df, kd_tree, points)
